In [0]:
import sys
sys.path.append('..')
sys.path.append('../..')

import lib_etl.validations_ETL as validations
from lib_etl.s3 import etl_input_data_validator
from lib.s3 import etl_input_table_validator
from lib.job_manager import load_config, split_config
from pyspark.sql import functions as f
from pyspark.sql.functions import col, concat, lpad

In [0]:
%run ../../config/utils

In [0]:
config = load_config(etl_config_path)
data_paths, club_square_config, config_validation = split_config(config)
run_as_date = dbutils.widgets.get("run_as_date")

### Transform 

In [0]:
member_tract_old = spark.table(bronze_member_tract)

acq_member_geo = spark.table(bronze_member_geo_archive)
max_date = acq_member_geo.orderBy(f.desc("file_modification_time")).first()["date"]
print(f'Latest modified date: {max_date}')

In [0]:
member_geo = acq_member_geo.filter(f.col("date") == "20240529")

member_tract = member_geo.withColumn("fips_state_code_padded", lpad(member_geo["fips_state_code"].cast("string"), 2, "0")) \
              .withColumn("fips_country_code_padded", lpad(member_geo["fips_country_code"].cast("string"), 3, "0")) \
              .withColumn("geo_census_2010_tract_padded", lpad(member_geo["geo_census_2010_tract"].cast("string"), 6, "0")) \
              .withColumn("TRACT", concat("fips_state_code_padded", "fips_country_code_padded", "geo_census_2010_tract_padded"))

member_tract = member_tract.withColumnRenamed("TRACT", "CENSUS_TRACT")
member_tract = member_tract.withColumnRenamed("mbr_sid", "MBRSHP_SID")
# In some cases 1 member has 2 rows in member_tract, and we need to drop them
# to prevent duplicate rows.
member_tract = member_tract.dropDuplicates(["MBRSHP_SID"])

In [0]:
validations.validate_table(
        spark, "source", "member_tract", config_validation, member_tract_old, stats_etl_path
    )

In [0]:
distance = spark.table(bronze_distance)

distance = distance.withColumnRenamed("TRACT", "CENSUS_TRACT")

census_tract = member_tract.join(distance, "CENSUS_TRACT", "left_outer")
census_tract.createOrReplaceTempView("census_tract")

print("JOIN SUCCESSFUL WITH CENSUS TRACT AND DISTANCE TABLES")


df_census_tract = spark.sql("""
select
    CENSUS_TRACT
    ,cast(zip as string) as ZIP
    ,cast(MBRSHP_SID as long) as MBRSHP_SID
    ,cast(BJS_Longitude as double) as LONGITUDE
    ,cast(BJs_Latitude as double) as LATITUDE
    ,cast(BJS_TRACT_LAT	 as double) as TRACT_LATITUDE
    ,cast(BJS_TRACT_LON as double) as TRACT_LONGITUDE
    ,cast(BJS_Distance as double) as BJS_DISTANCE
    ,cast(BJS_Driving_Distance as double) as BJS_DRIVING_DISTANCE
    ,cast(BJS_Drive_Time as double) as BJS_DRIVE_TIME
    ,cast(COSTCO_Distance as double) as COSTCO_DISTANCE
    ,cast(COSTCO_Driving_Distance as double) as COSTCO_DRIVING_DISTANCE
    ,cast(COSTCO_Drive_Time as double) as COSTCO_DRIVE_TIME
    ,cast(SAMS_Distance as double) as SAMS_DISTANCE
    ,cast(SAMS_Driving_Distance as double) as SAMS_DRIVING_DISTANCE
    ,cast(Sams_Drive_Time as double) as SAMS_DRIVE_TIME
    ,cast(Walmart_Distance as double) as WALMART_DISTANCE
    ,cast(Walmart_Driving_Distance as double) as WALMART_DRIVING_DISTANCE
    ,cast(Walmart_Drive_Time as double) as WALMART_DRIVE_TIME
    ,cast(ZIP_DISTANCE as double) as ZIP_DISTANCE
from census_tract
""")
df_census_tract = df_census_tract.dropDuplicates()

df_census_tract.createOrReplaceTempView("source")

In [0]:
validations.validate_table(
        spark, "intermediate", 'census_tract', config_validation, df_census_tract, stats_etl_path
    )

### Merge

In [0]:
df_census_tract.write.mode("overwrite").saveAsTable(silver_master_census_tract)

if archive_flag:
    save_archive(df_census_tract, silver_master_census_tract_archive, run_as_date)